# 数据读取 （data from Unit03_1_5_select.mat）

## 原始数据的读取

In [1]:
import scipy.io
import numpy as np

# 读取 .mat 文件
mat_data = scipy.io.loadmat('/home/charles/HZU/Data_processed/multi-condition-transfer-learning/Unit03_1_5_select_1.mat')

# 输出所有键（变量名），查看文件包含的内容
print("Keys in the .mat file:", mat_data.keys())

# 访问实际数据
data = mat_data['Unit03_1_5_select_1']

# 查看数据
print("Original data shape:", data.shape)  # (N, 14)

# ==================== 2️⃣ 设置抽样参数 ====================
n = 5000        # ⭐你只需要改这里
seed = 42      # 可选：保证可复现

np.random.seed(seed)

num_samples = data.shape[0]
assert n <= num_samples, "n cannot be larger than total samples!"

# ==================== 3️⃣ 随机抽取 n 个样本（行） ====================
indices = np.random.choice(num_samples, size=n, replace=False)
data = data[indices, :]

print("Sampled data shape:", data.shape)

# ==================== 4️⃣ 查看抽样结果 ====================
print(data[:5])   # 前 5 行看看

Keys in the .mat file: dict_keys(['__header__', '__version__', '__globals__', 'Unit03_1_5_select_1'])
Original data shape: (10000, 14)
Sampled data shape: (5000, 14)
[[ 6.10836744e+00  1.00559959e+01  2.04665691e-01 -1.36746206e+01
   1.13660057e+02  3.08465454e+02  5.95667358e+02  8.23152710e+02
   8.05135742e+02  8.77531250e+02  7.51735840e+02  7.52414917e+02
   7.50555908e+02  8.34698105e+00]
 [ 2.95663381e+00  5.93493752e-17  3.38581158e-04 -1.04039841e+01
   8.95254211e+01  2.96965881e+02  5.26934692e+02  6.48398132e+02
   6.05977295e+02  6.89841125e+02  7.01432434e+02  7.41026794e+02
   7.14204224e+02  1.05892191e+01]
 [ 8.98073101e-17  5.93493752e-17  2.74785707e-04 -2.13949895e+00
   2.68231030e+01  1.71628532e+01  1.93397350e+01  1.86293964e+01
   1.87852936e+01  1.95019188e+01  2.15198822e+01  1.91465244e+01
   2.33259640e+01  2.01965752e+01]
 [ 4.24416494e+00  5.93493752e-17  3.31420219e-04 -8.18939877e+00
   9.04087677e+01  3.05405426e+02  5.48547058e+02  6.96337585e+02
   

## 特征和标签的分离

In [2]:
import numpy as np

# 假设 'data' 是一个二维数组或矩阵
# 分离特征和标签

# 特征是除了最后一列的数据
X = data[:, :-1]  # 所有行，去除最后一列

# 标签是最后一列的数据
y = data[:, -1]  # 所有行，只取最后一列
y = y.reshape(-1, 1)

# # 查看特征和标签
# print("Features (X):")
# print(X[:5])  # 查看前5个特征样本
# print("Labels (y):")
# print(y[:5])  # 查看前5个标签

# 查看特征和标签的形状
print("Shape of Features (X):", X.shape)
print("Shape of Labels (y):", y.shape)

Shape of Features (X): (5000, 13)
Shape of Labels (y): (5000, 1)


## 三集划分

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split

# 假设 X 和 y 是已经分离好的特征和标签
# X: 特征数据，y: 标签数据

# 设置随机种子，确保结果可复现
random_seed = 42

# 控制三集的划分比例：例如 70% 训练集，15% 验证集，15% 测试集
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# 确保划分比例之和为1
assert train_ratio + val_ratio + test_ratio == 1.0, "The sum of ratios must be 1."

# 第一次划分，将训练集和验证+测试集合并
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=random_seed)

# 第二次划分，将验证集和测试集分开
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=test_ratio / (val_ratio + test_ratio), random_state=random_seed)

# 打印各个数据集的形状
print("Shape of Training Set (X_train, y_train):", X_train.shape, y_train.shape)
print("Shape of Validation Set (X_val, y_val):", X_val.shape, y_val.shape)
print("Shape of Test Set (X_test, y_test):", X_test.shape, y_test.shape)


Shape of Training Set (X_train, y_train): (3499, 13) (3499, 1)
Shape of Validation Set (X_val, y_val): (750, 13) (750, 1)
Shape of Test Set (X_test, y_test): (751, 13) (751, 1)


# 图结构搭建

In [4]:
import numpy as np
import torch
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# 输入：
#   X : numpy.ndarray or torch.Tensor
#       shape = [num_samples, num_features]
# 输出：
#   adj : torch.Tensor
#       shape = [num_features, num_features]
# =========================================================

def build_feature_graph_from_X(X, threshold=0.8, device="cpu"):
    """
    Automatically build a feature (column-wise) graph from X.

    Parameters
    ----------
    X : np.ndarray or torch.Tensor
        Shape [num_samples, num_features]
    threshold : float
        Cosine similarity threshold for edge creation
    device : str or torch.device
        cpu / cuda

    Returns
    -------
    adj : torch.Tensor
        Adjacency matrix of shape [num_features, num_features]
    """

    # ---------- 1️⃣ 统一成 numpy ----------
    if isinstance(X, torch.Tensor):
        X_np = X.detach().cpu().numpy()
    else:
        X_np = X

    # ---------- 2️⃣ 自动读取形状 ----------
    num_samples, num_features = X_np.shape
    print(f"[INFO] X shape: samples={num_samples}, features={num_features}")

    # ---------- 3️⃣ 特征（列）作为节点 ----------
    # 每一列是一个节点向量（跨样本）
    X_feature = X_np.T                         # [F, M]

    # ---------- 4️⃣ 特征间相似度 ----------
    sim_matrix = cosine_similarity(X_feature) # [F, F]

    # ---------- 5️⃣ 构建 NetworkX 图 ----------
    G = nx.Graph()
    G.add_nodes_from(range(num_features))

    for i in range(num_features):
        for j in range(i + 1, num_features):
            if sim_matrix[i, j] >= threshold:
                G.add_edge(i, j, weight=sim_matrix[i, j])

    print(f"[INFO] Graph built: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")

    # ---------- 6️⃣ Graph → 邻接矩阵 ----------
    adj_np = nx.to_numpy_array(G, weight="weight")  # [F, F]

    # ---------- 7️⃣ 转成 torch.Tensor ----------
    adj = torch.tensor(adj_np, dtype=torch.float32, device=device)

    # ---------- 8️⃣ 简单健壮性检查 ----------
    isolated = (adj.sum(dim=1) == 0).sum().item()
    if isolated > 0:
        print(f"[WARN] {isolated} isolated feature nodes detected "
              f"(consider lowering threshold or using KNN graph)")

    print(f"[INFO] adj shape: {adj.shape}")
    return adj


device = "cuda" if torch.cuda.is_available() else "cpu"

adj = build_feature_graph_from_X(X, threshold=0.8, device=device)

print(adj)


[INFO] X shape: samples=5000, features=13
[INFO] Graph built: nodes=13, edges=64
[WARN] 1 isolated feature nodes detected (consider lowering threshold or using KNN graph)
[INFO] adj shape: torch.Size([13, 13])
tensor([[0.0000, 0.8746, 0.8777, 0.0000, 0.9230, 0.9453, 0.9687, 0.9838, 0.9858,
         0.9847, 0.9794, 0.9800, 0.9808],
        [0.8746, 0.0000, 0.9984, 0.0000, 0.9137, 0.0000, 0.8127, 0.8517, 0.8570,
         0.8543, 0.8249, 0.8255, 0.8273],
        [0.8777, 0.9984, 0.0000, 0.0000, 0.9167, 0.0000, 0.8163, 0.8550, 0.8603,
         0.8577, 0.8287, 0.8293, 0.8311],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.9230, 0.9137, 0.9167, 0.0000, 0.0000, 0.8999, 0.9115, 0.9293, 0.9303,
         0.9316, 0.9152, 0.9138, 0.9158],
        [0.9453, 0.0000, 0.0000, 0.0000, 0.8999, 0.0000, 0.9953, 0.9854, 0.9828,
         0.9845, 0.9865, 0.9855, 0.9849],
        [0.9687, 0.8127, 0.8163, 0.0000, 0.9115, 0.9

# 模型

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ======================================================
# 0️⃣ 固定随机种子
# ======================================================
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================================================
# 1️⃣ 工具函数：统一转成 torch.Tensor
# ======================================================
def to_tensor(x, device):
    if isinstance(x, np.ndarray):
        return torch.tensor(x, dtype=torch.float32, device=device)
    elif isinstance(x, torch.Tensor):
        return x.to(device)
    else:
        raise TypeError(f"Unsupported type: {type(x)}")

# ======================================================
# 2️⃣ GCN Layer（adj = F × F）
# ======================================================
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x, adj):
        """
        x   : [F, in_dim]
        adj : [F, F]
        """
        F_dim = adj.shape[0]

        I = torch.eye(F_dim, device=adj.device)
        A_hat = adj + I

        deg = A_hat.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg, -0.5)
        deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
        D_inv_sqrt = torch.diag(deg_inv_sqrt)

        A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt
        out = A_norm @ x
        out = self.linear(out)
        return out

# ======================================================
# 3️⃣ 样本级 GCN 回归模型
# ======================================================
class SampleLevelGCNRegressor(nn.Module):
    def __init__(self, hidden_dim=64, out_dim=1):
        super().__init__()

        self.gcn1 = GCNLayer(1, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, hidden_dim)

        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, X_batch, adj):
        """
        X_batch : [B, F]
        adj     : [F, F]
        """
        B, F_dim = X_batch.shape
        outputs = []

        for b in range(B):
            x = X_batch[b].unsqueeze(1)      # [F,1]

            h = F.relu(self.gcn1(x, adj))    # [F,d]
            h = F.relu(self.gcn2(h, adj))    # [F,d]

            g = h.mean(dim=0)                # [d]
            y_hat = self.regressor(g)        # [out_dim]
            outputs.append(y_hat)

        return torch.stack(outputs, dim=0)   # [B,out_dim]

# ======================================================
# 4️⃣ 训练 / 验证函数
# ======================================================
def train_epoch(model, optimizer, criterion, X_data, y_data, adj, batch_size):
    model.train()
    num_samples = X_data.shape[0]
    perm = torch.randperm(num_samples, device=X_data.device)

    total_loss = 0.0

    for start in range(0, num_samples, batch_size):
        idx = perm[start:start + batch_size]
        Xb = X_data[idx]
        yb = y_data[idx]

        y_hat = model(Xb, adj)
        loss = criterion(y_hat, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * Xb.shape[0]

    return total_loss / num_samples


@torch.no_grad()
def eval_epoch(model, criterion, X_data, y_data, adj, batch_size):
    model.eval()
    num_samples = X_data.shape[0]
    total_loss = 0.0

    for start in range(0, num_samples, batch_size):
        Xb = X_data[start:start + batch_size]
        yb = y_data[start:start + batch_size]

        y_hat = model(Xb, adj)
        loss = criterion(y_hat, yb)

        total_loss += loss.item() * Xb.shape[0]

    return total_loss / num_samples

# ======================================================
# 6️⃣ 数据转 Tensor（关键！）
# ======================================================
X_train = to_tensor(X_train, device)
X_val   = to_tensor(X_val, device)
X_test  = to_tensor(X_test, device)

y_train = to_tensor(y_train, device)
y_val   = to_tensor(y_val, device)
y_test  = to_tensor(y_test, device)

adj = to_tensor(adj, device)

# 防呆检查
assert X_train.shape[1] == adj.shape[0], \
    f"Feature mismatch: X has {X_train.shape[1]}, adj is {adj.shape}"

# ======================================================
# 7️⃣ 训练配置
# ======================================================
model = SampleLevelGCNRegressor(hidden_dim=64, out_dim=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

batch_size = 32
epochs = 50

# ======================================================
# 8️⃣ 正式训练
# ======================================================
best_val = float("inf")
best_state = None

for epoch in range(1, epochs + 1):
    train_loss = train_epoch(
        model, optimizer, criterion,
        X_train, y_train, adj, batch_size
    )
    val_loss = eval_epoch(
        model, criterion,
        X_val, y_val, adj, batch_size
    )

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 5 == 0:
        print(f"[Epoch {epoch:03d}] "
              f"Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")

# ======================================================
# 9️⃣ 测试集评估
# ======================================================
model.load_state_dict(best_state)
model.to(device)

test_loss = eval_epoch(
    model, criterion,
    X_test, y_test, adj, batch_size
)

print(f"\n✅ Test MSE: {test_loss:.4f}")


[Epoch 001] Train MSE: 145.0464 | Val MSE: 140.0421
[Epoch 005] Train MSE: 86.0455 | Val MSE: 82.5267
[Epoch 010] Train MSE: 23.6835 | Val MSE: 21.2524
[Epoch 015] Train MSE: 10.6772 | Val MSE: 12.4988
[Epoch 020] Train MSE: 7.4568 | Val MSE: 7.2601
[Epoch 025] Train MSE: 5.6667 | Val MSE: 5.1942
[Epoch 030] Train MSE: 4.2978 | Val MSE: 4.0160
[Epoch 035] Train MSE: 3.5898 | Val MSE: 3.5679
[Epoch 040] Train MSE: 3.2854 | Val MSE: 3.5728
[Epoch 045] Train MSE: 3.0161 | Val MSE: 3.3020
[Epoch 050] Train MSE: 3.0283 | Val MSE: 2.8542

✅ Test MSE: 2.8508


In [6]:
save_path = "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/Single_condition_Regression/result/model_save/best_gcn_model.pth"
torch.save(best_state, save_path)
print(f"Model saved to {save_path}")


Model saved to /home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/Single_condition_Regression/result/model_save/best_gcn_model.pth


# 测试

In [8]:
from sklearn.metrics import r2_score
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===================== 1️⃣ 定义 R² 评估函数 =====================
@torch.no_grad()
def evaluate_r2(model, X_data, y_data, adj, batch_size):
    model.eval()

    y_true_list = []
    y_pred_list = []

    num_samples = X_data.shape[0]

    for start in range(0, num_samples, batch_size):
        Xb = X_data[start:start + batch_size].to(device)
        yb = y_data[start:start + batch_size].to(device)

        y_hat = model(Xb, adj)  # [B, 1]

        # 拉平成一维
        y_true_list.append(yb.view(-1).cpu().numpy())
        y_pred_list.append(y_hat.view(-1).cpu().numpy())

    y_true = np.concatenate(y_true_list, axis=0)
    y_pred = np.concatenate(y_pred_list, axis=0)

    r2 = r2_score(y_true, y_pred)
    return r2


# ===================== 2️⃣ 读取模型权重 =====================
MODEL_PATH = (
    "/home/charles/HZU/Industrial_Software_Testing/"
    "Industrial_Software_Testing/multi_condition_transfer_learning/"
    "Single_condition_Regression/result/model_save/"
    "best_gcn_model.pth"
)

# ⚠️ 强烈建议：显式 map_location
state_dict = torch.load(MODEL_PATH, map_location=device)

model.load_state_dict(state_dict)
model.to(device)

print("✅ Model loaded successfully")


# ===================== 3️⃣ 测试集 R² 评估 =====================
r2 = evaluate_r2(
    model,
    X_test,
    y_test,
    adj.to(device),
    batch_size=batch_size
)

print(f"✅ Test R²: {r2:.4f}")


/tmp/ipykernel_1990/676769253.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(MODEL_PATH, map_location=device)


✅ Model loaded successfully
✅ Test R²: 0.9045
